<a href="https://colab.research.google.com/github/Innovatewithapple/ResumeJD-CompressionPipeline/blob/main/ResumeJDCompressionPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers

In [ ]:
import torch
from datasets import load_dataset
import pandas as pd
import os
from collections import Counter
from transformers import AutoTokenizer,AutoModelForCausalLM
from nltk.tokenize import sent_tokenize
import time
import nltk
nltk.download('punkt_tab')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
resume_chunk_token = 1400
jd_chunk_token = 1000
resume_max_token = 600
jd_max_token = 250

In [ ]:
from huggingface_hub import login
login(token="")

In [ ]:
#--------Decoder-------!
decoder_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", trust_remote_code=True)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct",dtype=torch.float16,device_map="auto",trust_remote_code=True)

In [ ]:
#--------Data-Set-------!
ds = load_dataset("cnamuangtoun/resume-job-description-fit")

resume_texts = ds['train']['resume_text']
jd_texts = ds['train']['job_description_text']
label_text = ds['train']['label']

val_resume_texts = ds['test']['resume_text']
val_jd_texts = ds['test']['job_description_text']
val_label_text = ds['test']['label']

In [ ]:
print(Counter(label_text))
df = pd.DataFrame({"resume": resume_texts,"jd": jd_texts,"label": label_text})

fit_df = df[df["label"] == "Good Fit"].sample(100, random_state=42)

potential_df = df[df["label"] == "Potential Fit"].sample(100, random_state=42)

nofit_df = df[df["label"] == "No Fit"].sample(100, random_state=42)

balanced_df = pd.concat([
    fit_df,
    potential_df,
    nofit_df
]).sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
output_file = "compressed_dataset.csv"

if not os.path.exists(output_file):
    pd.DataFrame(
        columns=["resume","job_description","label"]
    ).to_csv(output_file,index=False)

In [ ]:
#----------Process Sent-tokenize and chunking-------!
def Chunk_process(text,token_size):
  sentences = sent_tokenize(text)
  sent_token_counts = [len(decoder_tokenizer.encode(sent,add_special_tokens=False)) for sent in sentences]

  chunks = []
  current_chunk = []
  current_len = 0

  for sent,token_len in zip(sentences,sent_token_counts):
    if current_len + token_len <= token_size:
      current_chunk.append(sent) # because sent itself a senetence not character so no need to use " "
      current_len += token_len
    else:
      if current_chunk:
        chunks.append(" ".join(current_chunk))
      current_chunk = [sent]
      current_len = token_len

  if current_chunk:
    chunks.append(" ".join(current_chunk))

  return chunks

In [ ]:
def Extract_Info(chunk,max_token,prompt):
  message = [
      {"role":"user","content":f"""{prompt}
{chunk}"""}
  ]
  prompt = decoder_tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)

  inputs = decoder_tokenizer(prompt,return_tensors='pt').to(llm.device)
  input_length = inputs["input_ids"].shape[1]


  with torch.no_grad():
    outputs = llm.generate(
        **inputs,
        max_new_tokens = max_token,
        do_sample=False,
        temperature=0.2,
        top_p=0.9,
        pad_token_id=decoder_tokenizer.eos_token_id
    )
  generated_tokens = outputs[0][input_length:]
  decoded = decoder_tokenizer.decode(generated_tokens,skip_special_tokens=True) #outputs has [batch,sequence],skip_special_tokens: when we dont want unneccesory tokens in our output strings, set true
  return decoded

In [ ]:
#-------Pre-process & Extract info--------!
def final_data(resume_texts,jd_texts,labels):
    output_file = "compressed_resume_jd_train.csv"

    if not os.path.exists(output_file):
        pd.DataFrame(
            columns=["resume","job_description","label"]
        ).to_csv(output_file,index=False)

    start_time = time.time()
    total_rows = len(resume_texts)

    for idx,(resume,jd,label) in enumerate(zip(resume_texts,jd_texts,labels)):
        #process chunk
        resume_process = Chunk_process(resume,resume_chunk_token)
        jd_process = Chunk_process(jd,jd_chunk_token)

        #llm process
        compressed_resume = Extract_Info(resume_process,resume_max_token,resume_prompt)
        compressed_jd = Extract_Info(jd_process,jd_max_token,job_description_prompt)

        temp_df = pd.DataFrame([{"resume": compressed_resume,"job_description": compressed_jd,"label": label }])
        temp_df.to_csv(output_file,mode="a",header=False,index=False)

        print(f"Saved {idx+1}")

        elapsed = time.time() - start_time
        avg_time_per_row = elapsed / (idx + 1)
        remaining_rows = total_rows - (idx + 1)
        eta_seconds = avg_time_per_row * remaining_rows
        eta_minutes = eta_seconds / 60

        print(f"Processed {idx+1}/{total_rows} | "f"Avg: {avg_time_per_row:.1f}s/row | "f"ETA: {eta_minutes:.1f} min")

    print("Finished")

In [ ]:
final_data(balanced_df["resume"],balanced_df["jd"],balanced_df["label"])

In [ ]:
#Below code use when session crash, and get extisting data
# existing_df = pd.read_csv("compressed_resume_jd_train.csv")

# already_done = len(existing_df)

# print(already_done)
# balanced_df = balanced_df.iloc[already_done:]

In [ ]:
#-----------Resume-Prompt---------!
resume_prompt = """You are an expert recruiter.

Create a concise candidate profile for hiring purposes.

Rules:

Use only information explicitly present in the resume.
Do NOT invent information.
Do NOT use outside knowledge.
Do NOT estimate missing details.
If information for a section is not explicitly present in the resume, return "Not explicitly stated".
Never infer missing values or missing achievements.
Never generate placeholders such as "[insert missing information]", "[not provided]", [not explicitly stated], "[unknown]", or "[TBD]".
Preserve important responsibilities, skills, technologies, leadership responsibilities, measurable achievements, education, and certifications.
Preserve important numbers, metrics, team sizes, budgets, revenues, assets managed, project scale, performance improvements, licenses, awards, and certifications when present.
Rank information by importance.
Keep only the most important information.
Maximum 3-5 bullet points per section.
Do NOT repeat information across sections.
Exclude generic, repetitive, or low-value details.
Use exact facts from the resume whenever possible.
Do NOT merge unrelated facts into a single statement.
Do NOT use subjective or promotional language.
Do NOT use phrases such as "highly motivated", "results-driven", "proven track record", "dynamic professional", or similar recruiting language.
Use factual statements only.

Return exactly:

Professional Summary:

Brief factual summary of the candidate's background and expertise in 2-3 sentences.
Focus on experience, expertise, industries, and responsibilities.
Do NOT include subjective descriptions.

Career History:

Include only professional employment positions.
Include job title and employment dates when explicitly available.
Do NOT include volunteer work, board memberships, extracurricular activities, sports, awards, licenses, certifications, education, or non-employment activities.
Maximum 10 entries.

Experience:

Summarize the candidate's most important responsibilities and work areas across their career.
Do NOT list every job separately unless necessary.
Focus on expertise, responsibilities, scope, and domain experience.
Maximum 5 bullet points.

Core Competencies:

List only the 5-10 most important professional competencies.
Prioritize technical, functional, and domain-specific competencies.
Include soft skills only if they are repeatedly emphasized or central to the candidate's work.
Do NOT include company names, job titles, generic words, incomplete phrases, or repeated information.

Software & Tools:

List only software, technologies, platforms, programming languages, databases, frameworks, business systems, or tools explicitly mentioned in the resume.
Do NOT include responsibilities, business functions, or general skills.

Leadership & Scope:

List only leadership, management, mentoring, coordination, supervision, or organizational responsibilities explicitly stated in the resume.
Do NOT infer responsibilities or scope.

Major Achievements:

List only measurable achievements, awards, certifications, licenses, project outcomes, business impact, or notable accomplishments explicitly stated in the resume.
Preserve important numbers and metrics whenever available.
Do NOT infer achievements that are not explicitly stated.

Education & Certifications:

List degrees, certifications, licenses, and formal education explicitly mentioned.

Resume:
"""

In [ ]:
#--------Job-description Prompt---------!
job_description_prompt = """You are an expert recruiter.

Extract only information useful for candidate-job matching.

Rules:

* Use only information explicitly stated in the job description.
* Do NOT invent information.
* Do NOT use outside knowledge.
* Do NOT infer missing requirements.
* Do NOT estimate missing experience, skills, education, certifications, or responsibilities.
* Remove company descriptions, company history, culture statements, benefits, equal opportunity statements, recruiter signatures, contact information, awards, accolades, marketing content, and promotional text.
* Remove duplicate responsibilities, skills, technologies, certifications, domains, and requirements.
* Preserve important requirements, responsibilities, skills, technologies, tools, domain knowledge, certifications, education, experience requirements, location, work mode, and employment type.
* Keep only information relevant for evaluating candidate fit.
* If information for a section is not explicitly present, return "Not specified".

Return exactly:

Role:

Extract the exact role title explicitly stated in the job description.
Do NOT shorten, rewrite, or generalize the title.

Experience Required:

List required years of experience and any experience requirements.

Key Responsibilities:

List only the most important responsibilities relevant to evaluating candidate fit.
Maximum 5 bullet points.

Required Skills:

List functional, business, analytical, technical, and domain skills explicitly required.
Maximum 10 items.

Required Technologies & Tools:

List software, databases, platforms, programming languages, frameworks, systems, or tools explicitly required.
Maximum 10 items.

Domain Knowledge:

List industries, business domains, regulations, methodologies, or subject-matter knowledge explicitly required.

Education:

List required or preferred education explicitly stated.

Certifications:

List required or preferred certifications explicitly stated.

Location & Work Mode:

List location and work arrangement (Remote, Hybrid, Onsite).

Employment Type:

List employment type (Full-time, Contract, Internship, Part-time, etc.).

Nice to Have:

List preferred qualifications, preferred skills, preferred certifications, or preferred experience explicitly stated.

Job Description:

"""